# Measure the Running Time of the Numerical Methods solving Stochastic Differential Games on Graphs

In [ ]:
'''
Num of players and graphs to get results for
'''
N_PLAYER_test_lst = [32, 64, 128]
graph_name_test_lst = ['complete']#,'star','cycle','hypercube',
                  #'complete bipartite','RSG','Ramanujan']

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import math
from math import isclose
import torch
import sys
import networkx as nx
import os
import scipy
import datetime
import random

Since this piece of code does not involve deep learning, it always runs on the CPU.

In [ ]:
device = torch.device('cpu')
print("Running on the CPU!")

## Model Settings and Parameters

There are $N$ players in the game with the dynamics given by
$$dX^i_t = \left[a\left(\frac{1}{\sqrt{d_{v_i}}}\sum_{j:v_j\sim v_i}\frac{1}{\sqrt{d_{v_j}}}X^j_t - X^i_t\right) + \alpha^i_t\right]\,dt + \sigma\,dW_t^i$$
where $a$ is the mean-reverting parameter, $d_{v_i}$ is the degree of vertex $v_i$ in the graph, $\alpha^i_t$ is the control of player $i$ at time $t$ and $(W^0_t,...,W^{N-1}_t)$ is an $N$-dimensional Brownian motion. The $\sim$ relationship between two vertices $v_i,v_j$ in the graph means that there exists an edge connecting $v_i$ to $v_j$. The running cost of player $i$ in this problem is set as
$$f^i(X_t,\alpha^i_t) = \frac{1}{2}(\alpha^i_t)^2 - q\alpha^i_t\left(\frac{1}{\sqrt{d_{v_i}}}\sum_{j:v_j\sim v_i}\frac{1}{\sqrt{d_{v_j}}}X^j_t - X^i_t\right) + \frac{\varepsilon}{2}\left(\frac{1}{\sqrt{d_{v_i}}}\sum_{j:v_j\sim v_i}\frac{1}{\sqrt{d_{v_j}}}X^j_t - X^i_t\right)^2$$
the terminal cost of player $i$ is set as
$$g^i(X_T) = \frac{c}{2}\left(\frac{1}{\sqrt{d_{v_i}}}\sum_{j:v_j\sim v_i}\frac{1}{\sqrt{d_{v_j}}}X^j_T - X^i_T\right)^2$$
with $a,\sigma,q,\varepsilon,c>0$ as parameters. We want to derive the Markovian Nash equilibrium where each player minimizes its expected cost
$$J^i(\alpha) = \mathbb{E}\left[\int_0^Tf^i(X_t,\alpha^i_t)\,dt + g^i(X_T)\right]$$
with time horizon $[0,T]$.

In [ ]:
'''
Model parameters
'''
N_PLAYER = 50
MOD_a = 0.1
MOD_sigma = 0.5
MOD_q = 1.0
MOD_eps = 1.0
MOD_c = 1.0
MOD_T = 1.0
MOD_DELTA = 1.0 # Initial state value U(-delta,delta)

In [ ]:
'''
FP stopping
'''
MAE_TOL = 1e-5 # Tolerance for MAE

The graph structure is encoded in the graph Laplacian $L$. Here we require the graph to be simple, connected and undirected, but not necessarily transitive. The graph Laplacian is defined as follows:
$$L\in\mathbb{R}^{N\times N},L_{ij} = \begin{cases}1 & i=j\\ -\frac{1}{\sqrt{d_{v_i}d_{v_j}}} & v_i\sim v_j\\ 0 & \text{else}\end{cases}$$

In [ ]:
'''
The following function returns the Laplacian of some commonly appearing graphs
sparsity index only used when the generated graph is random
it's organized as |E|/\binom{|V|}{2}, the number of edges over the number of all possible potential edges
'''
graph_name_lst = ['complete','star','cycle','petersen','hypercube','circulant',
                  'complete bipartite','RSG','Ramanujan']

def set_graph_LAP(name,graph_name_lst,sparsity_index = 0.125, degree = 6):
    assert (name in graph_name_lst), 'Graph name error!'
        
    if name == 'complete':
        G = nx.complete_graph(range(N_PLAYER))
        LAP = nx.normalized_laplacian_matrix(G).toarray()
    elif name == 'star':
        G = nx.star_graph(range(N_PLAYER))
        LAP = nx.normalized_laplacian_matrix(G).toarray()
    elif name == 'cycle':
        G = nx.cycle_graph(range(N_PLAYER))
        LAP = nx.normalized_laplacian_matrix(G).toarray() 
    elif name == 'petersen':
        assert N_PLAYER == 10, 'Error! Petersen graph must have 10 vertices!'
        G = nx.petersen_graph()
        LAP = nx.normalized_laplacian_matrix(G).toarray() 
    elif name == 'hypercube':
        k = int(np.round(np.log(N_PLAYER) / np.log(2)))
        assert 2 ** k == N_PLAYER, 'Error! N is not a power of 2!'
        G = nx.hypercube_graph(k)
        LAP = nx.normalized_laplacian_matrix(G).toarray() 
    elif name == 'circulant':
        G = nx.circulant_graph(N_PLAYER,offsets = [11])
        LAP = nx.normalized_laplacian_matrix(G).toarray()
    elif name == 'complete bipartite':
        m = N_PLAYER // 2
        assert 2 * m == N_PLAYER, 'Number of player is odd!'
        G = nx.complete_bipartite_graph(m, m)
        LAP = nx.normalized_laplacian_matrix(G).toarray()
    elif name == 'RSG':
        # Random sparse graph with the indicated sparsity index
        # Generate a complete graph
        G = nx.complete_graph(range(N_PLAYER))
        
        # Get a random spanning tree (uniform) and relabel all the nodes
        G = nx.random_spanning_tree(G)
        G = nx.convert_node_labels_to_integers(G)
        
        # Add edges until it meets the required sparsity
        exp_edge_num = int(np.round(N_PLAYER * (N_PLAYER - 1) / 2 * sparsity_index))
        current_edge_num = N_PLAYER - 1
        while current_edge_num < exp_edge_num:
            # Add a random edge (make sure the graph is simple)
            i = np.random.randint(0,N_PLAYER)
            j = np.random.randint(0,N_PLAYER)
            if i == j or G.has_edge(i,j):
                continue
            else:
                G.add_edge(i,j)
                current_edge_num = current_edge_num + 1   
        # Laplacian
        LAP = nx.normalized_laplacian_matrix(G).toarray()
    elif name == 'Ramanujan':
        # Fix seeds
        np.random.seed(42)
        random.seed(42)
        G = nx.random_regular_expander_graph(N_PLAYER, d = degree, epsilon = 0)
        # Laplacian
        LAP = nx.normalized_laplacian_matrix(G).toarray()
        
    # Save the graph plot
    nx.draw_networkx(G,with_labels = True)
    plt.savefig('plots/graph_structure.jpg',dpi = 600)
    plt.close()
    return LAP

In [ ]:
# Graph structure, LAP (N_PLAYER * N_PLAYER)
graph_name = 'star' # What's the name of the graph
LAP = set_graph_LAP(graph_name,graph_name_lst)

# If to build semi-explicit NE for transitive graphs
if (graph_name in ['complete','cycle','petersen','hypercube','circulant']
    and isclose(MOD_eps,MOD_q ** 2)):
    TRANSITIVE = True
else:
    TRANSITIVE = False

Since we need to numerically simulate the SDE solution with Euler scheme, it's good to specify how we are discretizing the time interval $[0,T]$ and how many samples we are using in Monte Carlo to approximate the expected cost.

In [ ]:
'''
Time discretization
'''
N_STEPS = 50 # Num of time steps in time discretization when plotting
N_STEPS_EXACT = 1000 # Num of time steps in time discretization when numerically solving ODEs

In [ ]:
# N_STEPS_EXACT must be integer multiple of N_STEPS
assert (N_STEPS_EXACT >= N_STEPS and N_STEPS_EXACT % N_STEPS == 0), 'Error! N_STEPS_EXACT is not a multiple of N_STEPS!'

The number trajectories we take to average when we compute the expected costs.

In [ ]:
'''
Evaluation and test setting
'''
N_SIM_TEST = 10  # Expected cost calculated based on how many samples
N_FIXED_BM = 1 # How many BM trajectories to fix when plotting
assert (N_FIXED_BM <= 10), 'Too many trajectories to plot!'

The error tolerance in FP, indicating when to stop.

In [ ]:
'''
FP stopping criterion
'''
FP_TOL = 1e-4

Fix a random seed when plotting the trajectories so it's possible to compare the performance of different set of hyperparameters when doing fine tuning.

In [ ]:
'''
Random seeds
'''
TORCH_INIT_SEED = 42 # Seed for initial state in testing
TORCH_PLOT_SEED = 100 # Seed for plotting

## Graph Information

The main information we are using is the degree and neighbors of each vertex in the graph.

In [ ]:
'''
Get the degrees and neighbors of each vertex in the graph
Input: Graph Laplacian (N_PLAYER * N_PLAYER)
Output: degree list (N_PLAYER), a list of list indicating neighbors
'''
def get_graph_info(LAP):
    N_PLAYER = LAP.shape[0]
    
    deg_lst = list() # Degree list
    nb_lst_lst = list() # neighbor list (a list of list)
    
    for vertex_i in range(N_PLAYER):
        LAP_col = LAP[:,vertex_i]
        nb_lst = list()
        deg_count = 0
        for vertex_j in range(N_PLAYER):
            # Do not count itself
            if vertex_i == vertex_j:
                continue
            if not isclose(LAP_col[vertex_j],0):
                # An edge detected
                deg_count = deg_count + 1
                nb_lst.append(vertex_j)
        # Record
        deg_lst.append(deg_count)
        nb_lst_lst.append(nb_lst)
    return deg_lst, nb_lst_lst

Get the degree list and the neighbor list that will be used later. Notice that the neighbor list is a list of list, its $i$-th entry is a list containing all neighbors of player $i$.

In [ ]:
# Get the information of the graph (degree list, neighbor list)
DEG_LST, NB_LST_LST = get_graph_info(LAP)

# Check if graph is connected
assert (0 not in DEG_LST), 'Error! The graph has to be connected!'

## Dynamics

The following function computes the term $\left(\frac{1}{\sqrt{d_{v_i}}}\sum_{j:v_j\sim v_i}\frac{1}{\sqrt{d_{v_j}}}X^j_t - X^i_t\right)$ for player $i$.

In [ ]:
'''
Calculate mean-reverting term for a certain player
Input: X_tensor as 2-dim tensor (N_PLAYER * N_SIM), the index of player
Output: a 2-dim column tensor (N_SIM * 1)
'''

'''
# This is the version for the exact mean reversion in the nbhd
def mean_rev_term(X_tensor,NB_LST_LST,ply_ind):
    # All neighbors of that player
    nb_ply_ind = NB_LST_LST[ply_ind]
    
    # Maintain the dimension as 2-dim (N_SIM * 1)
    X_tensor_nb = X_tensor[nb_ply_ind,:]
    mean_tensor = torch.unsqueeze(torch.mean(X_tensor_nb,dim = 0),1)
    ply_ind_tensor = torch.unsqueeze(X_tensor[ply_ind,:],1)
    return mean_tensor - ply_ind_tensor
'''

# This is the version for the generic model above (square root of degree)
def mean_rev_term(X_tensor,NB_LST_LST,ply_ind):
    # All neighbors of that player and their square root degrees
    nb_ply_ind = NB_LST_LST[ply_ind]
    nb_sqrt_deg = torch.sqrt(torch.Tensor(DEG_LST)[nb_ply_ind]).to(device)
    
    # Maintain the dimension as 2-dim (N_SIM * 1)
    X_tensor_nb = X_tensor[nb_ply_ind,:]
    ply_sqrt_deg = np.sqrt(DEG_LST[ply_ind])
    mean_tensor = 1 / ply_sqrt_deg * torch.unsqueeze(torch.sum(X_tensor_nb / nb_sqrt_deg[:,None],dim = 0),1)
    ply_ind_tensor = torch.unsqueeze(X_tensor[ply_ind,:],1)
    return mean_tensor - ply_ind_tensor

Build up functions in the dynamics of a certain player. Here it's taken as a linear-quadratic game.

In [ ]:
'''
Input: X_tensor as 2-dim tensor (N_PLAYER * N_SIM), the index of the player, model parameters
control of the player as 2-dim tensor (N_SIM * 1)
Output: a 2-dim column tensor (N_SIM * 1)
'''
# Drift
def dynamics_b(X_tensor,ply_ind,NB_LST_LST,MOD_a,ctrl_ply_ind):
    mean_rev = mean_rev_term(X_tensor,NB_LST_LST,ply_ind)
    return MOD_a * mean_rev + ctrl_ply_ind

# Running cost rate 
def dynamics_f(X_tensor,ply_ind,NB_LST_LST,MOD_q,MOD_eps,ctrl_ply_ind):
    mean_rev = mean_rev_term(X_tensor,NB_LST_LST,ply_ind)
    return 0.5 * (ctrl_ply_ind ** 2) - MOD_q * ctrl_ply_ind * mean_rev + 0.5 * MOD_eps * (mean_rev ** 2)

# Terminal cost
def dynamics_g(X_tensor,ply_ind,NB_LST_LST,MOD_c):
    mean_rev = mean_rev_term(X_tensor,NB_LST_LST,ply_ind)
    return 0.5 * MOD_c * (mean_rev ** 2)

## Exact Solution to NE

### Solve Riccati Equation

The exact solution is derived by noticing that in graph LQ game the Markovian Nash equilibrium is attained by the control of player $i$ at time $t$ facing state vector $X_t$ that
$$\hat{\alpha}^i(t,X_t) = -qe_i^TLX_t - e_i^TF^i_tX_t$$
where $F^i:[0,T]\to \mathbb{S}^{N\times N}$ takes value as a symmetric matrix determined by the following matrix-valued Riccati equation with given terminal condition
$$\dot{F}^i_t + \sum_{k=1}^N [-(a+q)L-F^k_t]e_ke_k^TF^i_t + \sum_{k=1}^N F^i_te_ke_k^T[-(a+q)L-F^k_t] + (\varepsilon - q^2)Le_ie_i^TL + F^i_te_ie_i^TF^i_t = 0$$
$$F^i_T = cLe_ie_i^TL$$
we use the method in scipy package to numerically solve the coupled ODE system and get $F^1,...,F^N$.

In [ ]:
'''
The following function calculates the derivative of F^i, flattened as a vector and concatenated.
Input: t as time, F as vec(F^0),...,vec(F^{N-1}) concatenated together (of length N_PLAYER^3), model parameters
Output: the derivative of F (of length N_PLAYER^3)
'''
def Riccati(t,F_vec,MOD_a,MOD_q,LAP,MOD_eps,N_PLAYER,fund_mat_list):
    # Cut vector y into N_PLAYER pieces and reshape into matrices
    F_piece_list = np.array_split(F_vec,N_PLAYER)
    F_mat_list = [np.reshape(F_piece,(N_PLAYER,N_PLAYER)) for F_piece in F_piece_list]
    
    # Calculate derivative
    F_prime_mat_list = list()
    for i in range(N_PLAYER):
        F_prime_mat = np.zeros((N_PLAYER,N_PLAYER))
        # Riccati equation above
        for k in range(N_PLAYER):
            LAP_term = (MOD_a + MOD_q) * LAP + F_mat_list[k]
            F_prime_mat = F_prime_mat + ((LAP_term @ fund_mat_list[k] @ F_mat_list[i]) 
                                         + (F_mat_list[i] @ fund_mat_list[k] @ LAP_term))

        F_prime_mat = F_prime_mat - (MOD_eps - MOD_q ** 2) * (LAP @ fund_mat_list[i] @ LAP)
        F_prime_mat = F_prime_mat - F_mat_list[i] @ fund_mat_list[i] @ F_mat_list[i]
        
        # Append
        F_prime_mat_list.append(F_prime_mat)
    
    # Flatten back into a vector and concatenate
    F_prime_vec = np.concatenate([np.reshape(F_prime_mat,(-1,)) for F_prime_mat in F_prime_mat_list])
    
    return F_prime_vec

In [ ]:
'''
The following function returns standard basis column vector e_i in R^N
Output: a 2-dim column vector of shape N * 1 as nparray
'''
def get_std_basis(i,N):
    std_basis = np.zeros(N)
    std_basis[i] = 1.0
    std_basis = np.reshape(std_basis,(-1,1))
    return std_basis

In [ ]:
'''
The following function returns a list of fundamental matrices with only one non-zero entry.
The nonzero entry is 1 on the diagonal.
Output: a list of fundamental matrices, i.e. e_1e_1^T, e_2e_2^T, ..., e_Ne_N^T
'''
def get_fund_mat_list(N_PLAYER):
    fund_mat_list = list()
    for i in range(N_PLAYER):
        std_basis = get_std_basis(i,N_PLAYER) # Col vector
        fund_mat_list.append(std_basis @ np.transpose(std_basis))
    return fund_mat_list

In [ ]:
'''
The following function prepares terminal condition for the Riccati equation
'''
def term_cond_Riccati(LAP,MOD_c,N_PLAYER,fund_mat_list):
    # Terminal condition
    term_cond_mat_list = [MOD_c * (LAP @ fund_mat_list[i] @ LAP) for i in range(N_PLAYER)]
        
    # Flatten each metrix into a vector and concatenate
    term_cond_vec = np.concatenate([np.reshape(term_cond_mat,(-1,)) for term_cond_mat in term_cond_mat_list])
    return term_cond_vec

In [ ]:
'''
The following function returns the solution to the Riccati equation using methods above
Output: Riccati_F_rec as an (N_PLAYER * N_STEPS) array of matrices (N_PLAYER * N_PLAYER), 
for each player and each time step, record the F^i which is the numerical solution to the Riccati equation
'''
def solve_Riccati_eqn(N_PLAYER,N_STEPS,LAP,MOD_c,MOD_T,MOD_a,MOD_q,MOD_eps):
    # Terminal condition
    fund_mat_list = get_fund_mat_list(N_PLAYER)
    term_cond_vec = term_cond_Riccati(LAP,MOD_c,N_PLAYER,fund_mat_list)

    # Numerically solving matrix-valued ODE system
    F_SOLUTION = scipy.integrate.solve_ivp(Riccati, [MOD_T, 0], term_cond_vec, 
                                       args=(MOD_a,MOD_q,LAP,MOD_eps,N_PLAYER,fund_mat_list),
                                       t_eval = np.linspace(MOD_T,0,N_STEPS),method = 'DOP853')
    print('Riccati solved!')
    # Notice that F_SOLUTION.y is of shape N_PLAYER^3 * N_STEPS
    Riccati_F = F_SOLUTION.y
    
    # The solution is now presented backward in time, so we need to reverse the time
    Riccati_F = np.fliplr(Riccati_F)

    # Record the solution F as a list of matrices
    # For each player at each time step record a matrix F
    Riccati_F_rec = np.empty((N_PLAYER,N_STEPS),'O')

    for col_ind in range(Riccati_F.shape[1]):
        # At a fixed time point
        F_fixed_time = Riccati_F[:,col_ind]
    
        # Split into N_PLAYER arrays
        F_fixed_time_list = np.array_split(F_fixed_time,N_PLAYER)
    
        # Reshape each array in the list
        F_mat_fixed_time_list = [np.reshape(F_array,(N_PLAYER,N_PLAYER)) for F_array in F_fixed_time_list]
    
        # Record it
        for player_ind in range(N_PLAYER):
            Riccati_F_rec[player_ind,col_ind] = F_mat_fixed_time_list[player_ind]
    
    return Riccati_F_rec

### Construct Exact NE from the Solution

In [ ]:
'''
The following function calculates the exact NE at a fixed time step (at time num_dt * dt)
Input: X_MC_tensor as 2-dim tensor (N_PLAYER * N_SIM), model parameters, solution to the Riccati equation
Output: an array of 2-dim (N_SIM * 1) tensors denoting each player's control
'''
def get_exact_control(num_dt,N_PLAYER,N_SIM,MOD_q,X_MC_tensor,Riccati_F_rec,LAP):
    LAP_tensor = torch.Tensor(LAP).to(device)
    
    # An array of 2-dim tensors (N_SIM * 1)
    ctrl = np.empty(N_PLAYER,'O') 
    
    for player_ind in range(N_PLAYER):
        std_basis = get_std_basis(player_ind,N_PLAYER) # Col vector
        std_basis = torch.Tensor(std_basis).to(device)
        
        # Riccati solution
        F_tensor = torch.Tensor(Riccati_F_rec[player_ind,num_dt]).to(device)

        # Formula for NE based on F from the Riccati equation (1 * N_SIM) tensor
        player_ind_ctrl = (- MOD_q * (torch.transpose(std_basis,0,1) @ LAP_tensor @ X_MC_tensor) \
                           - (torch.transpose(std_basis,0,1) @ F_tensor @ X_MC_tensor))
        
        # Transpose to get (N_SIM * 1) tensor
        ctrl[player_ind] = torch.transpose(player_ind_ctrl,0,1)
    return ctrl

## Analytic Solution on Transitive Graphs when $\varepsilon = q^2$

On vertex-transitive graphs, we have proved that the Markovian NE can be constructed in the following steps when $\varepsilon = q^2$:
1. Solve the matrix-valued ODE $R'(t) = \frac{1}{c}{\text{Tr}} \left[Q'(R(t)) e^{-t(a+q)L}\right]e^{-t(a+q)L},\quad R(0) = 0$ where $Q(X) := \left[\det\left(I+cXL\right)\right]^{\frac{1}{N}}$.
2. Compute $P_t = (a+q)L + R'(T-t)cL[I + R(T-t)cL]^{-1}$.
3. Compute $F^i_t = \frac{1}{\frac{\text{Tr}(P_t) - (a+q)\text{Tr}(L)}{N}}[P_t - (a+q)L]e_ie_i^T [P_t - (a+q)L]$.
4. Compute $\hat{\alpha}^i(t,x) = -qe_i^T L x - e_i^T F^i_t x$.

In [ ]:
'''
The following function implements the derivative of the Q function.
Input: an N by N matrix X
Output: an N by N matrix Q'(X)
'''
def trans_Q_prime(X):
    # Compute Q(X)
    Q = ((np.linalg.det(np.eye(N_PLAYER) + MOD_c * (X @ LAP))) ** (1 / N_PLAYER))
    
    # Q'(X) = Q(X) * c / N * (I + cXL)^{-1} * L
    Q_prime = Q * MOD_c / N_PLAYER * (np.linalg.inv(np.eye(N_PLAYER) + MOD_c * (X @ LAP)) @ LAP)
    
    return Q_prime

In [ ]:
'''
The following function calculates R'(t) above in the ODE.
Input: t as time, R_vec as vec(R) (of length N_PLAYER^2), model parameters
Output: the derivative of R as a vector (of length N_PLAYER^2)
'''
def trans_R_derivative(t,R_vec):
    # Reshape as N * N matrix
    R_reshaped = np.reshape(R_vec,(N_PLAYER,N_PLAYER))
    
    # Matrix exponential
    mat_exp = scipy.linalg.expm(-t * (MOD_a + MOD_q) * LAP)
    
    # Compute Q'(R) as an N * N matrix
    R_prime = 1 / MOD_c * np.matrix.trace(trans_Q_prime(R_reshaped) @ mat_exp) * mat_exp
    
    # Reshape into a vector with N^2 entries
    R_prime_reshaped = np.reshape(R_prime,(-1,))
    
    return R_prime_reshaped

In [ ]:
'''
The following function returns the solution to the ODE above
Output: trans_R_rec as a size N_STEPS array of matrices (N_PLAYER * N_PLAYER), 
for each time step, record the numerical solution to the ODE
'''
def solve_transitive_ODE(N_PLAYER,N_STEPS):
    # Initial condition
    init_cond_vec = np.reshape(np.zeros((N_PLAYER,N_PLAYER)),(-1,))

    # Numerically solving matrix-valued ODE system
    R_SOLUTION = scipy.integrate.solve_ivp(trans_R_derivative, [0, MOD_T], init_cond_vec, 
                                       t_eval = np.linspace(0,MOD_T,N_STEPS),method = 'DOP853')

    # Notice that R_SOLUTION.y is of shape N_PLAYER^2 * N_STEPS
    trans_R = R_SOLUTION.y

    # Record the solution R as a list of matrices
    # For each time step record a matrix R
    trans_R_rec = np.empty(N_STEPS,'O')

    for col_ind in range(trans_R.shape[1]):
        # At a fixed time point
        R_fixed_time = trans_R[:,col_ind]
    
        # Reshape each array in the list
        R_fixed_time_reshaped = np.reshape(R_fixed_time,(N_PLAYER,N_PLAYER))
    
        # Record it
        trans_R_rec[col_ind] = R_fixed_time_reshaped
    
    return trans_R_rec

In [ ]:
'''
This function computes P based on the R solved
Output: trans_P_rec as a size N_STEPS array of matrices (N_PLAYER * N_PLAYER), 
for each time step
'''
def get_trans_P(N_PLAYER,N_STEPS):
    # Numerically solve for R
    trans_R_rec = solve_transitive_ODE(N_PLAYER,N_STEPS)
    
    # Compute P
    trans_P_rec = np.empty(N_STEPS,'O')
    for time_ind in range(N_STEPS):
        # Get R(T - t)
        trans_R_vec_rev = np.flip(trans_R_rec)
        trans_R_rev = trans_R_vec_rev[time_ind]
        
        # Get T - t
        t_eval_rev = np.flip(np.linspace(0,MOD_T,N_STEPS))
        t_rev = t_eval_rev[time_ind]
    
        # Get R'(T-t)
        trans_R_rev_reshaped = np.reshape(trans_R_rev,(-1,))
        R_prime_rev_vec = trans_R_derivative(t_rev,trans_R_rev_reshaped)
        R_prime_rev = np.reshape(R_prime_rev_vec,(N_PLAYER,N_PLAYER))
        
        # Compute P
        trans_P_rec[time_ind] = ((MOD_a + MOD_q) * LAP + MOD_c * (R_prime_rev
                @ LAP @ np.linalg.inv(np.eye(N_PLAYER) + MOD_c * trans_R_rev @ LAP)))
    return trans_P_rec

In [ ]:
'''
This function computes F for each player at each time step based on P
for transitive graphs.
Output: trans_F_rec as a size (N_PLAYER * N_STEPS) array of matrices (N_PLAYER * N_PLAYER), 
for each player at each time step 
'''
def get_trans_F(N_PLAYER,N_STEPS):
    # Get P at each time step
    trans_P_rec = get_trans_P(N_PLAYER,N_STEPS)
    
    # Record F for each player at each time step
    trans_F_rec = np.empty((N_PLAYER,N_STEPS),'O')
    fund_mat_list = get_fund_mat_list(N_PLAYER)
    for time_ind in range(N_STEPS):
        # Get P_t
        trans_P = trans_P_rec[time_ind]
        tau = (np.matrix.trace(trans_P) - (MOD_a + MOD_q) * np.matrix.trace(LAP)) / N_PLAYER
        tmp_mat = trans_P - (MOD_a + MOD_q) * LAP
        
        for ply_ind in range(N_PLAYER):
            trans_F_rec[ply_ind,time_ind] = (1 / tau) * (tmp_mat @ fund_mat_list[ply_ind] @ tmp_mat)
    return trans_F_rec

## Fictitious Play

In [ ]:
'''
The following function computes the maximum relative error for F.
Input: two size (N_PLAYER * N_STEPS) arrays of matrices (N_PLAYER * N_PLAYER), 
for each player at each time step, one as exact, the other as semi-explicit.
Ouput: MRE
'''
def compute_MAE_MRE(exact_F_rec,semi_F_rec,N_PLAYER,N_STEPS):
    MAE = 0
    MRE = 0
    for ply_ind in range(N_PLAYER):
        for time_ind in range(N_STEPS):
            diff_norm = np.linalg.norm(exact_F_rec[ply_ind,time_ind] - semi_F_rec[ply_ind,time_ind])
            exact_norm = np.linalg.norm(exact_F_rec[ply_ind,time_ind])
            rel_norm = diff_norm / exact_norm
            MAE = np.maximum(MAE,diff_norm)
            MRE = np.maximum(MRE,rel_norm)
    return MAE, MRE

In [ ]:
'''
The following function calculates the derivative of F^{i,k+1}, flattened as a vector and concatenated.
Input: t as time, F as vec(F^{0,k}),...,vec(F^{N-1,k}) concatenated together (of length N_PLAYER^3), model parameters,
the F matrices in the last FP stage
Output: the derivative of F^{i,k+1} (of length N_PLAYER^3)
'''
def FP_Riccati(t,F_vec,MOD_a,MOD_q,LAP,MOD_eps,N_PLAYER,N_STEPS,fund_mat_list,last_FP_F_rec):
    # Select the correct time slice of past F according to current time t
    time_ind = int(np.round(t / N_STEPS))
    last_FP_F_list = last_FP_F_rec[:,time_ind]
    
    # Cut vector y into N_PLAYER pieces and reshape into matrices
    F_piece_list = np.array_split(F_vec,N_PLAYER)
    F_mat_list = [np.reshape(F_piece,(N_PLAYER,N_PLAYER)) for F_piece in F_piece_list]
    
    # Calculate derivative
    F_prime_mat_list = list()
    for i in range(N_PLAYER):
        F_prime_mat = np.zeros((N_PLAYER,N_PLAYER))
        # Riccati equation above
        for j in range(N_PLAYER):
            if j == i:
                continue
            # Only for j not equal to i
            F_prime_mat = F_prime_mat + ((F_mat_list[i] @ fund_mat_list[j] @ last_FP_F_list[j]) 
                                         + (last_FP_F_list[j] @ fund_mat_list[j] @ F_mat_list[i]))
        # EPS - q^2 term
        F_prime_mat = F_prime_mat - (MOD_eps - MOD_q ** 2) * (LAP @ fund_mat_list[i] @ LAP)
        
        # F^{i,k+1}e_ie_i^TF^{i,k+1} term
        F_prime_mat = F_prime_mat + F_mat_list[i] @ fund_mat_list[i] @ F_mat_list[i]
        
        # two (a + q) terms
        F_prime_mat = F_prime_mat + (MOD_a + MOD_q) * (LAP @ F_mat_list[i] + F_mat_list[i] @ LAP)
        
        # Append
        F_prime_mat_list.append(F_prime_mat)
    
    # Flatten back into a vector and concatenate
    F_prime_vec = np.concatenate([np.reshape(F_prime_mat,(-1,)) for F_prime_mat in F_prime_mat_list])
    
    return F_prime_vec

In [ ]:
'''
The following function prepares terminal condition for the FP Riccati equation
'''
def term_cond_FP_Riccati(LAP,MOD_c,N_PLAYER,fund_mat_list):
    # Terminal condition
    term_cond_mat_list = [MOD_c * (LAP @ fund_mat_list[i] @ LAP) for i in range(N_PLAYER)]
        
    # Flatten each metrix into a vector and concatenate
    term_cond_vec = np.concatenate([np.reshape(term_cond_mat,(-1,)) for term_cond_mat in term_cond_mat_list])
    return term_cond_vec

In [ ]:
'''
The following function initializes F for FP procedure
'''
def init_FP_Riccati(N_PLAYER,N_STEPS):
    # Init start as a list of zero matrices
    init_mat_rec = np.empty((N_PLAYER,N_STEPS),'O')
    for ply_ind in range(N_PLAYER):
        for t in range(N_STEPS):
            #init_mat_rec[ply_ind,t] = np.zeros((N_PLAYER,N_PLAYER))
            init_mat_rec[ply_ind,t] = np.eye(N_PLAYER)
    return init_mat_rec

In [ ]:
'''
The following function returns the solution to the Riccati equation using methods above
Output: Record the whole procedure of FP, return FP_history_F_rec as a list of (N_PLAYER * N_STEPS) array of matrices (N_PLAYER * N_PLAYER), 
for each player and each time step.
'''
def solve_FP_Riccati(N_PLAYER,N_STEPS,LAP,MOD_c,MOD_T,MOD_a,MOD_q,MOD_eps,exact_F_rec):
    # Terminal condition
    fund_mat_list = get_fund_mat_list(N_PLAYER)
    term_cond_vec = term_cond_FP_Riccati(LAP,MOD_c,N_PLAYER,fund_mat_list)
    
    # Starts FP with zero matrices
    last_FP_F_rec = init_FP_Riccati(N_PLAYER,N_STEPS)
    
    # Record the whole history of FP to see how F changes
    FP_history_F_rec = list()
    
    # Start FP
    FP_count = 0
    while 1:
        
        # Record the FP history
        FP_history_F_rec.append(last_FP_F_rec)

        # Numerically solving recursive matrix-valued ODE system
        F_SOLUTION = scipy.integrate.solve_ivp(FP_Riccati, [MOD_T, 0], term_cond_vec, 
                                       args=(MOD_a,MOD_q,LAP,MOD_eps,N_PLAYER,N_STEPS,fund_mat_list,last_FP_F_rec),
                                       t_eval = np.linspace(MOD_T,0,N_STEPS),method = 'DOP853')

        # Notice that F_SOLUTION.y is of shape N_PLAYER^3 * N_STEPS
        Riccati_F = F_SOLUTION.y
    
        # The solution is now presented backward in time, so we need to reverse the time
        Riccati_F = np.fliplr(Riccati_F)

        # Record the solution F as a list of matrices
        # For each player at each time step record a matrix F
        Riccati_F_rec = np.empty((N_PLAYER,N_STEPS),'O')

        for col_ind in range(Riccati_F.shape[1]):
            # At a fixed time point
            F_fixed_time = Riccati_F[:,col_ind]
    
            # Split into N_PLAYER arrays
            F_fixed_time_list = np.array_split(F_fixed_time,N_PLAYER)
    
            # Reshape each array in the list
            F_mat_fixed_time_list = [np.reshape(F_array,(N_PLAYER,N_PLAYER)) for F_array in F_fixed_time_list]
    
            # Record it
            for player_ind in range(N_PLAYER):
                Riccati_F_rec[player_ind,col_ind] = F_mat_fixed_time_list[player_ind]
        
        # Terminating criterion (MAE between two iterations is smaller than TOL)
        MAE, _ = compute_MAE_MRE(Riccati_F_rec,last_FP_F_rec,N_PLAYER,N_STEPS_EXACT)
        if MAE < MAE_TOL:
            break
                
        # Update the last_FP_F_list and enter the next iteration
        last_FP_F_rec = Riccati_F_rec
        FP_count += 1
        
    return FP_history_F_rec

## Compare the running time of Riccati, FP, and Semi-explicit.

Conduct experiments for different values of $N$, for all kinds of graphs mentioned above.

In [ ]:
fd = open('running_time.text','w')

In [ ]:
for N_PLAYER in N_PLAYER_test_lst:
    fd.write('\n')
    fd.write('Num of players: ' + str(N_PLAYER) + '\n')
    for graph_name in graph_name_test_lst:
        # Laplacian
        fd.write('\n')
        fd.write(graph_name + ':\n')
        LAP = set_graph_LAP(graph_name,graph_name_lst)

        # If to build semi-explicit NE
        if (graph_name in ['complete','cycle','petersen','hypercube','circulant','complete bipartite']
            and isclose(MOD_eps,MOD_q ** 2)):
            TRANSITIVE = True
        else:
            TRANSITIVE = False
            
        # Baseline
        start_time = datetime.datetime.now()
        Riccati_F_rec = solve_Riccati_eqn(N_PLAYER,N_STEPS_EXACT,LAP,MOD_c,MOD_T,MOD_a,MOD_q,MOD_eps)
        end_time = datetime.datetime.now()
        fd.write('\n')
        fd.write('Running time of numerically solving Riccati: ' + str(end_time - start_time) + '\n')
        
        # Semi-explicit
        if TRANSITIVE:
            start_time = datetime.datetime.now()
            trans_F_rec = get_trans_F(N_PLAYER,N_STEPS_EXACT)
            end_time = datetime.datetime.now()
            fd.write('\n')
            fd.write('Running time of semi-explicit construction: ' + str(end_time - start_time) + '\n')
            
        # FP
        start_time = datetime.datetime.now()
        FP_history_F_rec = solve_FP_Riccati(N_PLAYER,N_STEPS_EXACT,LAP,MOD_c,MOD_T,MOD_a,MOD_q,MOD_eps,exact_F_rec = Riccati_F_rec)
        end_time = datetime.datetime.now()
        fd.write('\n')
        fd.write('Running time of FP: ' + str(end_time - start_time) + '\n')  

## End of Code

In [ ]:
# Close the txt file
fd.close()

In [ ]:
# End of code
print('END OF CODE!!!')